In [0]:
import logging
import sys
import os

def setup_logging(log_level=logging.INFO, log_dir=None):
    """
    Configure logging for the entire project
    
    Parameters:
    - log_level: Logging level (default: INFO)
    - log_dir: Optional log directory path
    """
    
    # Create formatter
    formatter = logging.Formatter(
        fmt='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    # Get root logger
    root_logger = logging.getLogger()
    root_logger.setLevel(log_level)
    
    # Remove existing handlers to avoid duplicates
    root_logger.handlers.clear()
    
    # Console Handler (stdout) - Always add this
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(log_level)
    console_handler.setFormatter(formatter)
    root_logger.addHandler(console_handler)
    
    # File Handler (if log_dir is provided)
    if log_dir:
        try:
            # Create directory if it doesn't exist
            os.makedirs(log_dir, exist_ok=True)
            
            # Create log file path with timestamp
            from datetime import datetime
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            log_file = os.path.join(log_dir, f"etl_pipeline_{timestamp}.log")
            
            # Add file handler
            file_handler = logging.FileHandler(log_file, mode='a')
            file_handler.setLevel(log_level)
            file_handler.setFormatter(formatter)
            root_logger.addHandler(file_handler)
            
            root_logger.info(f"✓ Logging to file: {log_file}")
        except Exception as e:
            root_logger.warning(f"Could not create file handler: {str(e)}")
            root_logger.warning("Continuing with console logging only")
    
    # Reduce noise from Spark/Py4J loggers
    logging.getLogger('py4j').setLevel(logging.WARNING)
    logging.getLogger('pyspark').setLevel(logging.WARNING)
    
    return root_logger